In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/crime_500k_sample.csv")

print("Shape:", df.shape)
df.head()

Shape: (500000, 22)


,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,8466079,HV142615,02/02/2012 03:08:00 PM,066XX S ST LAWRENCE AVE,1811,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,True,False,...,20.0,42.0,18,1181377.0,1861042.0,2012.0,02/10/2018 03:50:01 PM,41.773938,-87.610657,"(41.773938066, -87.610656815)"
1,10039292,HY229009,04/11/2015 05:30:00 PM,069XX S JUSTINE ST,1310,CRIMINAL DAMAGE,TO PROPERTY,APARTMENT,False,False,...,17.0,67.0,14,1167204.0,1858642.0,2015.0,02/10/2018 03:50:01 PM,41.767667,-87.662681,"(41.767667227, -87.662680715)"
2,8656424,HV331830,06/12/2012 09:21:00 PM,051XX W NORTH AVE,041A,BATTERY,AGGRAVATED: HANDGUN,PARKING LOT/GARAGE(NON.RESID.),False,False,...,37.0,25.0,04B,1141672.0,1910146.0,2012.0,02/04/2016 06:33:39 AM,41.909510,-87.754997,"(41.909509901, -87.754996996)"
3,1527209,G277177,05/13/2001 11:30:00 PM,044XX S INDIANA AV,0915,MOTOR VEHICLE THEFT,"TRUCK, BUS, MOTOR HOME",STREET,False,False,...,NaN,NaN,07,1178295.0,1875859.0,2001.0,08/17/2015 03:03:40 PM,41.814668,-87.621505,"(41.814667833, -87.621504967)"
4,1583223,G348885,06/12/2001 07:00:00 AM,060XX S FRANCISCO AV,0810,THEFT,OVER $500,RESIDENCE,False,True,...,NaN,NaN,06,1158046.0,1864276.0,2001.0,09/07/2021 03:41:02 PM,41.783319,-87.696096,"(41.783318792, -87.696096026)"


In [2]:
missing_values = df.isnull().sum()

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage':
    (missing_values/len(df))*100
})

missing_df = missing_df.sort_values(
    by='Missing Percentage',
    ascending=False
)

missing_df.head(15)

,Missing Count,Missing Percentage
Ward,34438,6.8876
Community Area,34380,6.8760
Location,5491,1.0982
Y Coordinate,5491,1.0982
Longitude,5491,1.0982
Latitude,5491,1.0982
X Coordinate,5491,1.0982
Location Description,888,0.1776
District,4,0.0008
IUCR,0,0.0000


In [6]:
# Ward
df['Ward'] = df['Ward'].fillna(
    df['Ward'].median()
)

# Community Area
df['Community Area'] = df[
    'Community Area'
].fillna(
    df['Community Area'].median()
)

# District
df['District'] = df[
    'District'
].fillna(
    df['District'].median()
)

# Latitude Longitude
df = df.dropna(
    subset=['Latitude','Longitude']
)

df['Location Description'] = (
    df['Location Description']
    .fillna('UNKNOWN')
)

In [4]:
duplicates = df.duplicated().sum()

print("Duplicate Records:", duplicates)

df.drop_duplicates(inplace=True)

print("Shape After Removing Duplicates:")
print(df.shape)

Duplicate Records: 1567
Shape After Removing Duplicates:
(492942, 22)


In [7]:
missing_values = df.isnull().sum()

missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage':
    (missing_values/len(df))*100
})

missing_df = missing_df.sort_values(
    by='Missing Percentage',
    ascending=False
)

missing_df.head(15)

,Missing Count,Missing Percentage
ID,0,0.0
Case Number,0,0.0
Date,0,0.0
Block,0,0.0
IUCR,0,0.0
Primary Type,0,0.0
Description,0,0.0
Location Description,0,0.0
Arrest,0,0.0
Domestic,0,0.0


In [8]:
df['Date'] = pd.to_datetime(
    df['Date']
)

C:\Users\HP\AppData\Local\Temp\ipykernel_28020\2103529368.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(


In [9]:
df['Hour'] = df['Date'].dt.hour

df['DayOfWeek'] = (
    df['Date']
    .dt.day_name()
)

df['Month'] = (
    df['Date']
    .dt.month
)

df['Year'] = (
    df['Date']
    .dt.year
)

In [10]:
df['Hour'].value_counts()

Hour
12    28263
19    27731
0     27648
20    27612
18    27032
21    26753
15    26186
22    25974
17    25618
14    24922
16    24798
13    23331
23    21835
11    21796
9     21118
10    20897
8     16394
1     15819
2     13317
7     11379
3     10911
4      8511
6      8143
5      6954
Name: count, dtype: int64

In [11]:
df['IsWeekend'] = (
    df['Date']
    .dt.dayofweek >= 5
).astype(int)

In [12]:
def get_season(month):

    if month in [12,1,2]:
        return "Winter"

    elif month in [3,4,5]:
        return "Spring"

    elif month in [6,7,8]:
        return "Summer"

    else:
        return "Fall"

df['Season'] = df['Month'].apply(
    get_season
)

In [13]:
print("="*50)
print("DATA QUALITY REPORT")
print("="*50)

print(
    f"Total Records : {len(df):,}"
)

print(
    f"Total Features : {df.shape[1]}"
)

print(
    f"Duplicate Records : {df.duplicated().sum()}"
)

print(
    f"Missing Values : {df.isnull().sum().sum()}"
)

DATA QUALITY REPORT
Total Records : 492,942
Total Features : 27
Duplicate Records : 0
Missing Values : 0


In [14]:
invalid_coords = df[
    (df['Latitude'] < 41.5) |
    (df['Latitude'] > 42.1) |
    (df['Longitude'] < -88.0) |
    (df['Longitude'] > -87.4)
]

print("Invalid Records:", len(invalid_coords))

Invalid Records: 9


In [15]:
df = df[
    (df['Latitude'] >= 41.5) &
    (df['Latitude'] <= 42.1) &
    (df['Longitude'] >= -88.0) &
    (df['Longitude'] <= -87.4)
]

In [16]:
print(df.shape)

(492933, 27)


In [17]:
print("Latitude Range")

print(
    df['Latitude'].min(),
    df['Latitude'].max()
)

print("Longitude Range")

print(
    df['Longitude'].min(),
    df['Longitude'].max()
)

Latitude Range
41.644604096 42.022644369
Longitude Range
-87.934272688 -87.524529378


In [18]:
df.to_csv(
    "../data/crime_cleaned.csv",
    index=False
)

print(
    "Cleaned dataset saved successfully"
)

Cleaned dataset saved successfully


In [19]:
df

,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,...,Year,Updated On,Latitude,Longitude,Location,Hour,DayOfWeek,Month,IsWeekend,Season
0,8466079,HV142615,2012-02-02 15:08:00,066XX S ST LAWRENCE AVE,1811,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,True,False,...,2012,02/10/2018 03:50:01 PM,41.773938,-87.610657,"(41.773938066, -87.610656815)",15,Thursday,2,0,Winter
1,10039292,HY229009,2015-04-11 17:30:00,069XX S JUSTINE ST,1310,CRIMINAL DAMAGE,TO PROPERTY,APARTMENT,False,False,...,2015,02/10/2018 03:50:01 PM,41.767667,-87.662681,"(41.767667227, -87.662680715)",17,Saturday,4,1,Spring
2,8656424,HV331830,2012-06-12 21:21:00,051XX W NORTH AVE,041A,BATTERY,AGGRAVATED: HANDGUN,PARKING LOT/GARAGE(NON.RESID.),False,False,...,2012,02/04/2016 06:33:39 AM,41.909510,-87.754997,"(41.909509901, -87.754996996)",21,Tuesday,6,0,Summer
3,1527209,G277177,2001-05-13 23:30:00,044XX S INDIANA AV,0915,MOTOR VEHICLE THEFT,"TRUCK, BUS, MOTOR HOME",STREET,False,False,...,2001,08/17/2015 03:03:40 PM,41.814668,-87.621505,"(41.814667833, -87.621504967)",23,Sunday,5,1,Spring
4,1583223,G348885,2001-06-12 07:00:00,060XX S FRANCISCO AV,0810,THEFT,OVER $500,RESIDENCE,False,True,...,2001,09/07/2021 03:41:02 PM,41.783319,-87.696096,"(41.783318792, -87.696096026)",7,Tuesday,6,0,Summer
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
499994,10924956,JA239414,2017-04-26 01:35:00,033XX N NEVA AVE,502P,OTHER OFFENSE,FALSE/STOLEN/ALTERED TRP,STREET,False,False,...,2017,02/10/2018 03:50:01 PM,41.940324,-87.805588,"(41.940324299, -87.805588284)",1,Wednesday,4,0,Spring
499995,13344301,JH122558,2024-01-20 16:28:00,006XX N MICHIGAN AVE,0860,THEFT,RETAIL THEFT,SMALL RETAIL STORE,False,False,...,2024,12/21/2024 03:40:46 PM,41.892753,-87.624194,"(41.892753031, -87.624193896)",16,Saturday,1,1,Winter
499996,7739020,HS547008,2010-10-04 08:50:00,056XX W ROSCOE ST,0460,BATTERY,SIMPLE,SIDEWALK,False,False,...,2010,02/10/2018 03:50:01 PM,41.942214,-87.767857,"(41.942213812, -87.76785674)",8,Monday,10,0,Fall
499997,9453097,HX106439,2014-01-07 23:10:00,023XX N CICERO AVE,1811,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,STREET,True,False,...,2014,02/04/2016 06:33:39 AM,41.923689,-87.746285,"(41.923689051, -87.746285109)",23,Tuesday,1,0,Winter
